In [ ]:
""" Here You can Transforma Data Using Mongo db to azure """

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, LongType, DateType
from pyspark.sql import functions as F

In [0]:
from urllib.parse import quote_plus

In [0]:
password = quote_plus("Password")
host = "Host"
port = "Port"
user_name = "Username"
database_name = "database_name"
collection_name = "collection_name"
url = f"mongodb://{user_name}:{password}@{host}:{port}/{database_name}.{collection_name}?authSource=admin"

In [0]:
df = spark.read\
    .format('com.mongodb.spark.sql.DefaultSource')\
    .option( "uri", url) \
    .load()

In [0]:
""" Avaibility Data Cleanup """
availability_keys = df.schema["availability"].dataType.names
availability_df = df.select(
    F.col("_id").alias("airbnb_id"),
    *[F.col(f"availability.{key}").alias(key) for key in availability_keys]
)

In [0]:
""" Images Data Cleanup """
images_keys = df.schema["images"].dataType.names
images_df = df.select(
    F.col("_id").alias("airbnb_id"),
    *[F.col(f"images.{key}").alias(key) for key in images_keys]
)

In [0]:
""" Address Data Cleanup """
address_keys = df.schema["address"].dataType.names
address_df = df.select(
    F.col("_id").alias("airbnb_id"),
    *[F.col(f"address.{key}").alias(key) for key in address_keys]
)

address_df = address_df.drop("location")

In [0]:
""" Host Data Cleanup """
host_keys = df.schema["host"].dataType.names
host_df = df.select(
    *[F.col(f"host.{key}").alias(key) for key in host_keys]
)
# host_df.withColumn("host_verifications", F.concat_ws(",", "host_verifications"))
host_df = address_df.drop("host_verifications")
df = df.withColumn('host_id', F.col('host').getItem('host_id'))


In [0]:
""" Review Data Cleanup """
exploded_df = df.select(F.explode(F.col("reviews")).alias("review"))
filtered_df = exploded_df.filter(F.col("review").isNotNull())
reviews_keys = filtered_df.schema["review"].dataType.names
reviews_df = filtered_df.select(
    *[F.col(f"review.{key}").alias(key) for key in reviews_keys]
)


In [0]:
""" Review Score Data Cleanup """
review_scores_keys = df.schema["review_scores"].dataType.names
review_scores_df = df.select(
    F.col("_id").alias("airbnb_id"),
    *[F.col(f"review_scores.{key}").alias(key) for key in review_scores_keys]
)

In [0]:
df = df.drop('host', 'reviews', 'address', 'images', 'availability', 'amenities', 'review_scores')


In [0]:
container_name = "container_name"
account_name = "account_name"
account_key = "<Account-Key>"

In [0]:
spark.conf.set(f"fs.azure.account.key.{account_name}.blob.core.windows.net", account_key)

In [0]:
des_url = f"wasbs://{container_name}@{account_name}.blob.core.windows.net/silver/airbnb/"


In [0]:
reviews_df.write.csv(des_url + "reviews_df.csv", mode='overwrite', header=True)
availability_df.write.csv(des_url + "availability_df.csv", mode='overwrite', header=True)
address_df.write.csv(des_url + "address_df.csv", mode='overwrite', header=True)
host_df.write.csv(des_url + "host_df.csv", mode='overwrite', header=True)
review_scores_df.write.csv(des_url + "review_scores_df.csv", mode='overwrite', header=True)
df.write.csv(des_url + "airbnb.csv", mode='overwrite', header=True)